In [63]:
import sys, os

sys.path.append(os.path.abspath("/Users/rems/Library/CloudStorage/OneDrive-ETHZurich/ETHz_HS25/MScThesis/code"))


from LagrangainArchitecture import *

In [64]:
from symbolica import *
from IPython.display import Markdown as md, display as dp
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets

def display(self) -> None:
    dp(md(self.to_latex().replace('gamma', '\\gamma').replace('mu', '\\mu')))

def display_graph(self: Graph) -> None:
    dp(md('```mermaid\n' + self.to_mermaid() + '\n```'))

In [65]:
phi = Field("phi", kind="scalar", self_conjugate=True)
psi = Field("psi", kind="fermion", indexed=True, self_conjugate=False)

y = Parameter("y")
mpsi = Parameter("mpsi")

print("Scalar field:", phi())
print("Fermion field:", psi(i, f))
print("Conjugate fermion:", psi.conjugate(i, f))
print("Parameter y:", y.symbol)
print("Parameter mpsi:", mpsi.symbol)

Scalar field: phi
Fermion field: psi(bis(4,i),flavor(3,f))
Conjugate fermion: psibar(bis(4,i),flavor(3,f))
Parameter y: y
Parameter mpsi: mpsi


In [66]:
expr_scalar_mass = mpsi.symbol * phi() * phi()
expr_fermion_mass = mpsi.symbol * psi.conjugate(i, f) * psi(i, f)
expr_yukawa = y.symbol * phi() * psi.conjugate(i, f) * psi(i, f)

print("Scalar mass term:")
print(expr_scalar_mass)
print()

print("Fermion mass term:")
print(expr_fermion_mass)
print()

print("Yukawa term:")
print(expr_yukawa)

Scalar mass term:
phi^2*mpsi

Fermion mass term:
mpsi*psi(bis(4,i),flavor(3,f))*psibar(bis(4,i),flavor(3,f))

Yukawa term:
phi*y*psi(bis(4,i),flavor(3,f))*psibar(bis(4,i),flavor(3,f))


In [67]:
expr = expr_yukawa

print("Expression:")
print(expr)
print()

print("Atom tree:")
print(expr.to_atom_tree())

Expression:
phi*y*psi(bis(4,i),flavor(3,f))*psibar(bis(4,i),flavor(3,f))

Atom tree:


In [68]:
print("All symbols:")
print(expr.get_all_symbols())

All symbols:
[spenso::bis, spenso::flavor, spynso3::i, spynso3::f, spenso_python::psi, spenso_python::psibar, phi, y]


In [69]:
print("All indeterminates:")
print(expr.get_all_indeterminates())

All indeterminates:
[spynso3::i, spynso3::f, phi, y, spenso::bis(4,spynso3::i), spenso::flavor(3,spynso3::f), spenso_python::psi(spenso::bis(4,spynso3::i),spenso::flavor(3,spynso3::f)), spenso_python::psibar(spenso::bis(4,spynso3::i),spenso::flavor(3,spynso3::f))]


# Canonical tensor experiment

In [70]:
phi_head = S("phi")
i_sym = S("i")
j_sym = S("j")
k_sym = S("k")

e1 = phi_head(i_sym) * phi_head(i_sym)
e2 = phi_head(j_sym) * phi_head(j_sym)
e3 = phi_head(k_sym) * phi_head(k_sym)

print("e1 =", e1)
print("e2 =", e2)
print("e3 =", e3)

e1 = phi(i)^2
e2 = phi(j)^2
e3 = phi(k)^2


In [71]:
expr1, extra1, map1 = e1.canonize_tensors([(i_sym, 0)])
expr2, extra2, map2 = e2.canonize_tensors([(j_sym, 0)])
expr3, extra3, map3 = e3.canonize_tensors([(k_sym, 0)])

d = S("d")

e1n = expr1.replace(i_sym, d)
e2n = expr2.replace(j_sym, d)
e3n = expr3.replace(k_sym, d)

print(e1n)
print(e2n)
print(e3n)
print("All equal?", e1n.to_canonical_string() == e2n.to_canonical_string() == e3n.to_canonical_string())

phi(d)^2
phi(d)^2
phi(d)^2
All equal? True


# Pattern matching playground: simplest version

In [72]:
expr = y.symbol * phi() * psi.conjugate(i, f) * psi(i, f)
print("Expression:")
print(expr)

Expression:
phi*y*psi(bis(4,i),flavor(3,f))*psibar(bis(4,i),flavor(3,f))


In [73]:
print(expr.to_atom_tree())
print(expr.get_all_symbols())

[spenso::bis, spenso::flavor, spynso3::i, spynso3::f, spenso_python::psi, spenso_python::psibar, phi, y]


# A first useful helper: detect whether a field head appears

In [74]:
def contains_head(expr, head_name: str) -> bool:
    symbols = {str(s) for s in expr.get_all_symbols()}
    return head_name in symbols

def symbol_names(expr):
    return sorted({str(s) for s in expr.get_all_symbols()})

print(symbol_names(expr_yukawa))
print(symbol_names(expr_fermion_mass))
print(symbol_names(expr_scalar_mass))

def classify_term(expr):
    names = symbol_names(expr)

    has_phi = "phi" in names
    has_psi = "psi" in names
    has_psibar = "psibar" in names

    if has_phi and has_psi and has_psibar:
        return "yukawa-like interaction"
    if has_psi and has_psibar and not has_phi:
        return "fermion bilinear"
    if has_phi and not has_psi and not has_psibar:
        return "scalar-only term"
    return "unknown"

['bis', 'f', 'flavor', 'i', 'phi', 'psi', 'psibar', 'y']
['bis', 'f', 'flavor', 'i', 'mpsi', 'psi', 'psibar']
['mpsi', 'phi']


In [75]:
print("Contains phi?", contains_head(expr_yukawa, "phi"))
print("Contains psi?", contains_head(expr_yukawa, "psi"))
print("Contains psibar?", contains_head(expr_yukawa, "psibar"))
print("Contains y?", contains_head(expr_yukawa, "y"))

Contains phi? True
Contains psi? True
Contains psibar? True
Contains y? True


# Derivative playground

In [76]:
delx = S("delx")
mu_sym = S("mu")

expr_deriv = delx(mu_sym) * phi()
print(expr_deriv)
print(expr_deriv.to_atom_tree())
print(symbol_names(expr_deriv))

phi*delx(mu)
['delx', 'mu', 'phi']


In [77]:
p = S("p")
ii = S("ii")   # temporary symbolic placeholder,

In [78]:
try:
    replaced = expr_deriv.replace(delx(mu_sym), -ii * p(mu_sym))
    print(replaced)
except Exception as e:
    print("Replacement failed:", e)

-phi*ii*p(mu)


# Easy start

In [79]:
phi = S("phi")
psi = S("psi")
psibar = S("psibar")

y = S("y")
m = S("m")
lam = S("lam")

i = S("i")
j = S("j")
k = S("k")
f = S("f")

bis = S("bis")
flavor = S("flavor")

In [80]:
psi_if = psi(bis(4, i), flavor(3, f))
psibar_if = psibar(bis(4, i), flavor(3, f))

print("psi_if =", psi_if)
print("psibar_if =", psibar_if)

psi_if = psi(bis(4,i),flavor(3,f))
psibar_if = psibar(bis(4,i),flavor(3,f))


In [81]:
L1 = m * phi * phi
L2 = m * psibar_if * psi_if
L3 = y * phi * psibar_if * psi_if
L4 = lam * phi * phi * phi * phi

print("L1 =", L1)
print("L2 =", L2)
print("L3 =", L3)
print("L4 =", L4)

L1 = phi^2*m
L2 = m*psi(bis(4,i),flavor(3,f))*psibar(bis(4,i),flavor(3,f))
L3 = phi*y*psi(bis(4,i),flavor(3,f))*psibar(bis(4,i),flavor(3,f))
L4 = phi^4*lam


### strip the coefficient from L3

In [82]:
expr = L3
print("expr =", expr)

expr1 = expr.replace(phi, 1)
print("after removing phi:", expr1)

expr2 = expr1.replace(psi_if, 1)
print("after removing psi:", expr2)

expr3 = expr2.replace(psibar_if, 1)
print("after removing psibar:", expr3)

expr = phi*y*psi(bis(4,i),flavor(3,f))*psibar(bis(4,i),flavor(3,f))
after removing phi: y*psi(bis(4,i),flavor(3,f))*psibar(bis(4,i),flavor(3,f))
after removing psi: y*psibar(bis(4,i),flavor(3,f))
after removing psibar: y


### strip the coefficient from L2

In [83]:
expr = L2
print("expr =", expr)

expr1 = expr.replace(psi_if, 1)
print("after removing psi:", expr1)

expr2 = expr1.replace(psibar_if, 1)
print("after removing psibar:", expr2)

expr = m*psi(bis(4,i),flavor(3,f))*psibar(bis(4,i),flavor(3,f))
after removing psi: m*psibar(bis(4,i),flavor(3,f))
after removing psibar: m


### canonicalize phi(i) phi(i) and phi(j) phi(j)

In [84]:
e1 = phi(i) * phi(i)
e2 = phi(j) * phi(j)

print("e1 =", e1)
print("e2 =", e2)

c1, extra1, map1 = e1.canonize_tensors([(i, 0), (j, 0), (k, 0)])
c2, extra2, map2 = e2.canonize_tensors([(j, 0), (i, 0), (k, 0)])

print("canon(e1) =", c1)
print("canon(e2) =", c2)

print("same canonical string?", c1.to_canonical_string() == c2.to_canonical_string())

e1 = phi(i)^2
e2 = phi(j)^2
canon(e1) = phi(i)^2
canon(e2) = phi(i)^2
same canonical string? True


### canonicalize a quartic scalar operator

In [85]:
e3 = phi(i) * phi(i) * phi(j) * phi(j)
e4 = phi(k) * phi(k) * phi(j) * phi(j)

print("e3 =", e3)
print("e4 =", e4)

c3, extra3, map3 = e3.canonize_tensors([(i, 0), (j, 0), (k, 0)])
c4, extra4, map4 = e4.canonize_tensors([(i, 0), (j, 0), (k, 0)])

print("canon(e3) =", c3)
print("canon(e4) =", c4)

print("same canonical string?", c3.to_canonical_string() == c4.to_canonical_string())

e3 = phi(i)^2*phi(j)^2
e4 = phi(j)^2*phi(k)^2
canon(e3) = phi(i)^2*phi(j)^2
canon(e4) = phi(i)^2*phi(j)^2
same canonical string? True


### replace the fermion bilinear by a symbol

In [86]:
B = S("B")
bilinear = psi_if * psibar_if

print("bilinear =", bilinear)

print("L3 ->", L3.replace(bilinear, B))
print("L2 ->", L2.replace(bilinear, B))

bilinear = psi(bis(4,i),flavor(3,f))*psibar(bis(4,i),flavor(3,f))
L3 -> phi*y*B
L2 -> m*B


In [87]:
expr = y * phi * psi_if * psibar_if + m * psi_if * psibar_if
print("expr =", expr)
print("expr with bilinear replaced =", expr.replace(bilinear, B))

expr = phi*y*psi(bis(4,i),flavor(3,f))*psibar(bis(4,i),flavor(3,f))+m*psi(bis(4,i),flavor(3,f))*psibar(bis(4,i),flavor(3,f))
expr with bilinear replaced = phi*y*B+m*B


In [88]:
print(expr3)   # from Exercise 1
print(expr2)   # from Exercise 2
print(c1.to_canonical_string())
print(c2.to_canonical_string())
print(L3.replace(bilinear, B))

y
m
(python::{}::phi(python::{}::i))^2
(python::{}::phi(python::{}::i))^2
phi*y*B


### coefficient of the fermion bilinear inside a sum

In [89]:
exprB = expr.replace(bilinear, B)
print(exprB.replace(B, 1))

phi*y+m


### coefficient of phi^2

In [101]:
A = S("A")
phi2 = phi * phi
phi4= phi * phi * phi * phi

expr = m * phi * phi + lam * phi * phi * phi * phi
print("expr =", expr)

exprA = expr.replace_multiple([
    Replacement(phi4, A),
    Replacement(phi2, A),
])

print("replace phi^2 -> A:", exprA)

expr = phi^2*m+phi^4*lam
replace phi^2 -> A: m*A+lam*A


### derivative to momentum

In [56]:
delx = S("delx")
mu = S("mu")
p = S("p")
ii = S("ii")

Dphi = delx(mu) * phi
print("Dphi =", Dphi)

print("replace derivative:", Dphi.replace(delx(mu), -ii * p(mu)))

Dphi = phi*delx(mu)
replace derivative: -phi*ii*p(mu)


In [57]:
expr = y * delx(mu) * phi * psi_if * psibar_if
print("expr =", expr)

expr_mom = expr.replace(delx(mu), -ii * p(mu))
print("after derivative -> momentum:", expr_mom)

expr = phi*y*delx(mu)*psi(bis(4,i),flavor(3,f))*psibar(bis(4,i),flavor(3,f))
after derivative -> momentum: -phi*y*ii*p(mu)*psi(bis(4,i),flavor(3,f))*psibar(bis(4,i),flavor(3,f))


### compare order of the bilinear

In [58]:
print((psi_if * psibar_if).to_canonical_string())
print((psibar_if * psi_if).to_canonical_string())
print((psi_if * psibar_if).to_canonical_string() == (psibar_if * psi_if).to_canonical_string())

python::{}::psi(python::{}::bis(4,python::{}::i),python::{}::flavor(3,python::{}::f))*python::{}::psibar(python::{}::bis(4,python::{}::i),python::{}::flavor(3,python::{}::f))
python::{}::psi(python::{}::bis(4,python::{}::i),python::{}::flavor(3,python::{}::f))*python::{}::psibar(python::{}::bis(4,python::{}::i),python::{}::flavor(3,python::{}::f))
True


### quartic operator recognition

In [109]:
expr = lam * phi(i) * phi(i) * phi(j) * phi(j)
print("expr =", expr)

c, extra, mp = expr.canonize_tensors([(i, 0), (j, 0),(k, 0)])
print("canon(expr) =", c)

expr = lam*phi(i)^2*phi(j)^2
canon(expr) = lam*phi(i)^2*phi(j)^2


In [110]:
expr2 = lam * phi(k) * phi(k) * phi(i) * phi(i)
c2, extra2, mp2 = expr2.canonize_tensors([(i, 0), (j, 0), (k, 0)])

print("canon(expr2) =", c2)
print(c.to_canonical_string() == c2.to_canonical_string())

canon(expr2) = lam*phi(i)^2*phi(j)^2
True
